In [ ]:
import json
from IPython.display import display, Markdown
from scipy.stats import spearmanr
from matplotlib_inline.backend_inline import set_matplotlib_formats

from emu_renewal.constants import DATA_PATH, FULL_RUN
from emu_renewal.inputs import get_world_shp
from emu_renewal.outputs import get_all_policy_effect_metrics
from emu_renewal.plotting import plot_effect_scatter, plot_param_map
from emu_renewal.utils import get_analysis_paths, get_analysis_commits_df

set_matplotlib_formats("svg")

In [ ]:
world = get_world_shp()
world["geometry"] = world.simplify(tolerance=0.1, preserve_topology=True)

all_countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))
analysis_paths = get_analysis_paths(FULL_RUN, all_countries)
metrics = get_all_policy_effect_metrics(analysis_paths)
world = world.merge(metrics, on="ISO_A3", how="left")

# Purpose
This document compares the realised magnitude of policy scaling
under the OxCGRT floored and independent analyses.
For each posterior draw, $\log M_t$ is reconstructed from the country's
own observed indicators.
The peak metric is the range of that series;
the mean metric is the range from the loosest observed setting
to the time-average.
Country values are posterior medians.
The two analyses are compared by Spearman rank correlation
and by maps of each metric.

In [ ]:
rows = []
for metric in ["peak", "mean"]:
    rho, p = spearmanr(metrics[f"floored_{metric}"], metrics[f"indep_{metric}"])
    rows.append(f"- {metric}: Spearman $\\rho$ = {rho:.2f} ($p$ = {p:.3g})")
display(Markdown("## Concordance\n\n" + "\n".join(rows)))
for metric in ["peak", "mean"]:
    display(plot_effect_scatter(metrics, metric))

In [ ]:
for metric in ["peak", "mean"]:
    display(Markdown(f"## {metric.capitalize()} policy effect"))
    for analysis, col in [
        ("OxCGRT floored", f"floored_{metric}"),
        ("OxCGRT independent", f"indep_{metric}"),
    ]:
        display(Markdown(f"### {analysis}"))
        plot_param_map(world, col, metrics[col].max(), title=analysis)

{{< pagebreak >}}

# Commits used for analyses
For reproducibility, the following table gives the (short) commit SHA for each analysis.

In [ ]:
Markdown(get_analysis_commits_df(analysis_paths).to_markdown())